# Modified Bessel Functions — Colab Test Suite

Tests `modified_bessel_i_forward(x, nu)` and `modified_bessel_k_forward(x, nu)` against SciPy.

**No full PyTorch build.** Clones repo for source, compiles only the bessel
functions as a tiny C++/CUDA extension (~2 min), tests on CPU and GPU.

1. **Runtime > Change runtime type > T4 GPU > Save**
2. **Runtime > Run all**

In [ ]:
#@title Cell 1: Install deps + clone repo (source only, no build)
!pip install scipy ninja -q
!git clone --branch feature/bessel-arbitrary-order --depth 1 \
    https://github.com/shc443/pytorch.git /content/pytorch-src 2>/dev/null || \
    echo 'Already cloned'
!ls /content/pytorch-src/aten/src/ATen/native/Math.h && echo 'Source files ready'

In [ ]:
#@title Cell 2: Compile bessel extension (CPU + CUDA, ~2 min)
import torch
from torch.utils.cpp_extension import load_inline

cpp_source = r"""
#include <torch/extension.h>
#include <ATen/native/Math.h>

torch::Tensor bessel_i_cpu(torch::Tensor x, torch::Tensor nu) {
    TORCH_CHECK(x.sizes() == nu.sizes(), "x and nu must have same shape");
    auto out = torch::empty_like(x);
    AT_DISPATCH_FLOATING_TYPES(x.scalar_type(), "bessel_i_cpu", [&] {
        auto x_ptr = x.data_ptr<scalar_t>();
        auto nu_ptr = nu.data_ptr<scalar_t>();
        auto out_ptr = out.data_ptr<scalar_t>();
        for (int64_t i = 0; i < x.numel(); i++) {
            out_ptr[i] = modified_bessel_i_forward(x_ptr[i], nu_ptr[i]);
        }
    });
    return out;
}

torch::Tensor bessel_k_cpu(torch::Tensor x, torch::Tensor nu) {
    TORCH_CHECK(x.sizes() == nu.sizes(), "x and nu must have same shape");
    auto out = torch::empty_like(x);
    AT_DISPATCH_FLOATING_TYPES(x.scalar_type(), "bessel_k_cpu", [&] {
        auto x_ptr = x.data_ptr<scalar_t>();
        auto nu_ptr = nu.data_ptr<scalar_t>();
        auto out_ptr = out.data_ptr<scalar_t>();
        for (int64_t i = 0; i < x.numel(); i++) {
            out_ptr[i] = modified_bessel_k_forward(x_ptr[i], nu_ptr[i]);
        }
    });
    return out;
}
"""

cuda_source = r"""
#include <ATen/native/Math.h>
#include <c10/cuda/CUDAException.h>

template<typename scalar_t>
__global__ void bessel_i_kernel(const scalar_t* x, const scalar_t* nu,
                                scalar_t* out, int64_t n) {
    int64_t idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < n) {
        out[idx] = modified_bessel_i_forward(x[idx], nu[idx]);
    }
}

template<typename scalar_t>
__global__ void bessel_k_kernel(const scalar_t* x, const scalar_t* nu,
                                scalar_t* out, int64_t n) {
    int64_t idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < n) {
        out[idx] = modified_bessel_k_forward(x[idx], nu[idx]);
    }
}

torch::Tensor bessel_i_cuda(torch::Tensor x, torch::Tensor nu) {
    TORCH_CHECK(x.sizes() == nu.sizes());
    auto out = torch::empty_like(x);
    int64_t n = x.numel();
    int threads = 256;
    int blocks = (n + threads - 1) / threads;
    AT_DISPATCH_FLOATING_TYPES(x.scalar_type(), "bessel_i_cuda", [&] {
        bessel_i_kernel<<<blocks, threads>>>(
            x.data_ptr<scalar_t>(), nu.data_ptr<scalar_t>(),
            out.data_ptr<scalar_t>(), n);
        C10_CUDA_KERNEL_LAUNCH_CHECK();
    });
    return out;
}

torch::Tensor bessel_k_cuda(torch::Tensor x, torch::Tensor nu) {
    TORCH_CHECK(x.sizes() == nu.sizes());
    auto out = torch::empty_like(x);
    int64_t n = x.numel();
    int threads = 256;
    int blocks = (n + threads - 1) / threads;
    AT_DISPATCH_FLOATING_TYPES(x.scalar_type(), "bessel_k_cuda", [&] {
        bessel_k_kernel<<<blocks, threads>>>(
            x.data_ptr<scalar_t>(), nu.data_ptr<scalar_t>(),
            out.data_ptr<scalar_t>(), n);
        C10_CUDA_KERNEL_LAUNCH_CHECK();
    });
    return out;
}
"""

bessel_ext = load_inline(
    name='bessel_ext',
    cpp_sources=[cpp_source],
    cuda_sources=[cuda_source],
    extra_include_paths=['/content/pytorch-src/aten/src'],
    functions=['bessel_i_cpu', 'bessel_k_cpu', 'bessel_i_cuda', 'bessel_k_cuda'],
    verbose=True,
)

def modified_bessel_i(x, nu):
    x, nu = torch.as_tensor(x), torch.as_tensor(nu)
    nu = nu.expand_as(x) if nu.dim() == 0 else nu
    x = x.expand_as(nu) if x.dim() == 0 else x
    if x.is_cuda:
        return bessel_ext.bessel_i_cuda(x.contiguous(), nu.contiguous())
    return bessel_ext.bessel_i_cpu(x.contiguous(), nu.contiguous())

def modified_bessel_k(x, nu):
    x, nu = torch.as_tensor(x), torch.as_tensor(nu)
    nu = nu.expand_as(x) if nu.dim() == 0 else nu
    x = x.expand_as(nu) if x.dim() == 0 else x
    if x.is_cuda:
        return bessel_ext.bessel_k_cuda(x.contiguous(), nu.contiguous())
    return bessel_ext.bessel_k_cpu(x.contiguous(), nu.contiguous())

# Smoke test
x = torch.tensor([1.0, 2.0, 3.0])
nu = torch.tensor([0.5, 1.0, 2.0])
print('I_nu(x):', modified_bessel_i(x, nu))
print('K_nu(x):', modified_bessel_k(x, nu))
print('\nExtension compiled OK!')

In [ ]:
#@title Cell 3: Test helpers
import scipy.special
import numpy as np
import time

def max_rel_error(ref, actual):
    ref = np.asarray(ref, dtype=np.float64)
    actual = np.asarray(actual, dtype=np.float64)
    mask = (ref != 0) & np.isfinite(ref) & np.isfinite(actual)
    if not mask.any(): return 0.0
    return float(np.max(np.abs((ref[mask] - actual[mask]) / ref[mask])))

class Results:
    def __init__(self):
        self.passed = self.failed = self.skipped = 0
        self.failures = []
    def ok(self, name, cond):
        if cond: self.passed += 1; print(f'  [PASS] {name}')
        else: self.failed += 1; self.failures.append(name); print(f'  [FAIL] {name}')
    def skip(self, name, r): self.skipped += 1; print(f'  [SKIP] {name} ({r})')
    def header(self, t): print(f"\n{'='*60}\n  {t}\n{'='*60}")
    def summary(self):
        t = self.passed + self.failed
        print(f"\n{'='*60}\n  RESULTS: {self.passed}/{t} passed, {self.failed} failed, {self.skipped} skipped\n{'='*60}")
        if self.failed == 0: print('  ALL TESTS PASSED')
        else:
            for f in self.failures: print(f'    - {f}')

R = Results()

In [ ]:
#@title Cell 4: Test 1 — Precision vs SciPy (CPU)
R.header('TEST 1: Precision vs SciPy (CPU)')

np.random.seed(42)
x_np = np.random.uniform(0.1, 20.0, size=500).astype(np.float64)
nu_np = np.random.uniform(-10.0, 10.0, size=500).astype(np.float64)

for dtype, tol in [(torch.float32, 1e-3), (torch.float64, 1e-5)]:
    np_dtype = {torch.float32: np.float32, torch.float64: np.float64}[dtype]
    x_t = torch.tensor(x_np, dtype=dtype)
    nu_t = torch.tensor(nu_np, dtype=dtype)

    ref_i = scipy.special.iv(nu_np, x_np).astype(np_dtype)
    out_i = modified_bessel_i(x_t, nu_t).numpy()
    err_i = max_rel_error(ref_i, out_i)
    R.ok(f'CPU I_nu {dtype} max_rel_err={err_i:.2e} (tol={tol})', err_i < tol)

    ref_k = scipy.special.kv(nu_np, x_np).astype(np_dtype)
    out_k = modified_bessel_k(x_t, nu_t).numpy()
    err_k = max_rel_error(ref_k, out_k)
    R.ok(f'CPU K_nu {dtype} max_rel_err={err_k:.2e} (tol={tol})', err_k < tol)

In [ ]:
#@title Cell 5: Test 2 — Edge cases
R.header('TEST 2: Edge cases')

v = modified_bessel_i(torch.tensor([0.0]), torch.tensor([0.0])).item()
R.ok(f'I_0(0) = {v:.4f} (expect 1.0)', abs(v - 1.0) < 1e-6)

v = modified_bessel_i(torch.tensor([0.0]), torch.tensor([2.0])).item()
R.ok(f'I_2(0) = {v:.6f} (expect 0.0)', abs(v) < 1e-6)

v = modified_bessel_i(torch.tensor([0.0]), torch.tensor([0.5])).item()
R.ok(f'I_0.5(0) = {v:.6f} (expect 0.0)', abs(v) < 1e-6)

x_t = torch.tensor([1.0, 2.0, 5.0], dtype=torch.float64)
i_pos = modified_bessel_i(x_t, torch.tensor([2.0, 2.0, 2.0], dtype=torch.float64))
i_neg = modified_bessel_i(x_t, torch.tensor([-2.0, -2.0, -2.0], dtype=torch.float64))
err = max_rel_error(i_pos.numpy(), i_neg.numpy())
R.ok(f'I_{{-2}}(x) == I_2(x), rel_err={err:.2e}', err < 1e-10)

v = modified_bessel_i(torch.tensor([-1.0]), torch.tensor([1.0])).item()
R.ok('I_1(-1) = NaN', np.isnan(v))

v = modified_bessel_k(torch.tensor([0.0]), torch.tensor([1.0])).item()
R.ok(f'K_1(0) = inf', v == float('inf') or v > 1e30)

k_pos = modified_bessel_k(x_t, torch.tensor([2.5, 2.5, 2.5], dtype=torch.float64))
k_neg = modified_bessel_k(x_t, torch.tensor([-2.5, -2.5, -2.5], dtype=torch.float64))
err = max_rel_error(k_pos.numpy(), k_neg.numpy())
R.ok(f'K_{{-2.5}}(x) == K_2.5(x), rel_err={err:.2e}', err < 1e-10)

v = modified_bessel_k(torch.tensor([-1.0]), torch.tensor([1.0])).item()
R.ok('K_1(-1) = NaN', np.isnan(v))

In [ ]:
#@title Cell 6: Test 3 — Large + fractional orders
R.header('TEST 3: Large and fractional orders')

v = modified_bessel_i(torch.tensor([10.0], dtype=torch.float64), torch.tensor([50.0], dtype=torch.float64)).item()
ref = scipy.special.iv(50.0, 10.0)
err = abs(v - ref) / abs(ref) if ref != 0 else abs(v)
R.ok(f'I_50(10) rel_err={err:.2e}', err < 1e-3)

v = modified_bessel_k(torch.tensor([10.0], dtype=torch.float64), torch.tensor([50.0], dtype=torch.float64)).item()
ref = scipy.special.kv(50.0, 10.0)
err = abs(v - ref) / abs(ref) if ref != 0 else abs(v)
R.ok(f'K_50(10) rel_err={err:.2e}', err < 1e-3)

x_f = torch.tensor([1.0, 3.0, 7.0], dtype=torch.float64)
nu_f = torch.tensor([2.5, 2.5, 2.5], dtype=torch.float64)
ref_i = scipy.special.iv(2.5, x_f.numpy())
out_i = modified_bessel_i(x_f, nu_f).numpy()
R.ok(f'I_2.5(x) rel_err={max_rel_error(ref_i, out_i):.2e}', max_rel_error(ref_i, out_i) < 1e-5)

ref_k = scipy.special.kv(2.5, x_f.numpy())
out_k = modified_bessel_k(x_f, nu_f).numpy()
R.ok(f'K_2.5(x) rel_err={max_rel_error(ref_k, out_k):.2e}', max_rel_error(ref_k, out_k) < 1e-5)

In [ ]:
#@title Cell 7: Test 4 — Precision vs SciPy (CUDA)
R.header('TEST 4: Precision vs SciPy (CUDA)')

if not torch.cuda.is_available():
    R.skip('CUDA precision', 'No CUDA')
else:
    np.random.seed(123)
    x_np = np.random.uniform(0.1, 20.0, size=500).astype(np.float64)
    nu_np = np.random.uniform(-10.0, 10.0, size=500).astype(np.float64)

    for dtype, tol in [(torch.float32, 1e-3), (torch.float64, 1e-5)]:
        np_dtype = {torch.float32: np.float32, torch.float64: np.float64}[dtype]
        x_t = torch.tensor(x_np, dtype=dtype, device='cuda')
        nu_t = torch.tensor(nu_np, dtype=dtype, device='cuda')

        ref_i = scipy.special.iv(nu_np, x_np).astype(np_dtype)
        out_i = modified_bessel_i(x_t, nu_t).cpu().numpy()
        R.ok(f'CUDA I_nu {dtype} rel_err={max_rel_error(ref_i, out_i):.2e}', max_rel_error(ref_i, out_i) < tol)

        ref_k = scipy.special.kv(nu_np, x_np).astype(np_dtype)
        out_k = modified_bessel_k(x_t, nu_t).cpu().numpy()
        R.ok(f'CUDA K_nu {dtype} rel_err={max_rel_error(ref_k, out_k):.2e}', max_rel_error(ref_k, out_k) < tol)

In [ ]:
#@title Cell 8: Test 5 — CPU/CUDA parity
R.header('TEST 5: CPU/CUDA parity')

if not torch.cuda.is_available():
    R.skip('CPU/CUDA parity', 'No CUDA')
else:
    np.random.seed(456)
    x_np = np.random.uniform(0.1, 20.0, size=500).astype(np.float64)
    nu_np = np.random.uniform(-10.0, 10.0, size=500).astype(np.float64)
    x_cpu = torch.tensor(x_np, dtype=torch.float64)
    nu_cpu = torch.tensor(nu_np, dtype=torch.float64)

    cpu_i = modified_bessel_i(x_cpu, nu_cpu)
    gpu_i = modified_bessel_i(x_cpu.cuda(), nu_cpu.cuda()).cpu()
    R.ok(f'I_nu parity rel_err={max_rel_error(cpu_i.numpy(), gpu_i.numpy()):.2e}', max_rel_error(cpu_i.numpy(), gpu_i.numpy()) < 1e-12)

    cpu_k = modified_bessel_k(x_cpu, nu_cpu)
    gpu_k = modified_bessel_k(x_cpu.cuda(), nu_cpu.cuda()).cpu()
    R.ok(f'K_nu parity rel_err={max_rel_error(cpu_k.numpy(), gpu_k.numpy()):.2e}', max_rel_error(cpu_k.numpy(), gpu_k.numpy()) < 1e-10)

    x32 = x_cpu.float(); nu32 = nu_cpu.float()
    cpu32 = modified_bessel_i(x32, nu32)
    gpu32 = modified_bessel_i(x32.cuda(), nu32.cuda()).cpu()
    R.ok(f'I_nu float32 parity rel_err={max_rel_error(cpu32.numpy(), gpu32.numpy()):.2e}', max_rel_error(cpu32.numpy(), gpu32.numpy()) < 1e-6)

In [ ]:
#@title Cell 9: Test 6 — CUDA stress test (1M elements)
R.header('TEST 6: CUDA stress test')

if not torch.cuda.is_available():
    R.skip('stress test', 'No CUDA')
else:
    N = 1_000_000
    x_big = torch.rand(N, dtype=torch.float32, device='cuda') * 20.0 + 0.1
    nu_big = torch.rand(N, dtype=torch.float32, device='cuda') * 20.0 - 10.0

    torch.cuda.synchronize(); t0 = time.perf_counter()
    out_i = modified_bessel_i(x_big, nu_big)
    torch.cuda.synchronize(); dt_i = time.perf_counter() - t0

    torch.cuda.synchronize(); t0 = time.perf_counter()
    out_k = modified_bessel_k(x_big, nu_big)
    torch.cuda.synchronize(); dt_k = time.perf_counter() - t0

    R.ok(f'I_nu 1M no crash, {dt_i:.3f}s', out_i.shape == (N,))
    R.ok(f'K_nu 1M no crash, {dt_k:.3f}s', out_k.shape == (N,))
    R.ok('I_nu no NaN', not torch.isnan(out_i).any().item())
    R.ok('K_nu no NaN (x>0.1)', not torch.isnan(out_k).any().item())
    R.ok('I_nu all finite', torch.isfinite(out_i).all().item())

In [ ]:
#@title Cell 10: Summary
R.summary()